# 01 – Conventions, scorer checks and data report
Checks the coordinate and rotation conventions against the official scorer and measures the dataset before modelling. CPU only.

**Results**
- 19/19 unit tests pass. The official scorer confirms the conventions: GT round-trip scores 1.0, the Rz(π) twin is free, a head↔feet flip costs 180°.
- Metric sensitivity: a single moderate error (0.2 m, 15 % size, 30° rotation) barely changes mAP3D; errors along the box's thin axis (range x for upright divers) cost the most IoU.
- Pose shift: train is 19.6 % upright vs 52.1 % in val (test ≈ 57 %) → pose-rebalanced sampling in 02.
- Consecutive frames are highly similar, frames are contiguous, divers move ~3–7 cm per frame → temporal smoothing in 04.

## Data

In [1]:
import os, sys, glob, subprocess
if not glob.glob("/tmp/data/**/annotations", recursive=True):
    subprocess.run("pip -q install gdown", shell=True, check=True)
    subprocess.run('gdown --folder "https://drive.google.com/drive/folders/16HHHfCWM_p8RIvHQkQ38PMnZJNXzgXdU" -O /tmp/dl', shell=True, check=True)
    subprocess.run("unzip -q /tmp/dl/diver3d_v2.zip -d /tmp/data", shell=True, check=True)
ROOT = glob.glob("/tmp/data/**/annotations", recursive=True)[0].rsplit("/annotations", 1)[0]
print("ROOT =", ROOT)

Retrieving folder contents
Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From (original): https://drive.google.com/uc?id=1MxBRohKXmK5ewLgNe773_SwmIqvTis_G
From (redirected): https://drive.google.com/uc?id=1MxBRohKXmK5ewLgNe773_SwmIqvTis_G&confirm=t&uuid=289e251a-4bcf-48fc-bdb2-6b946c312f42
To: /tmp/dl/diver3d_v2.zip
  0%|          | 0.00/3.45G [00:00<?, ?B/s]

Processing file 1MxBRohKXmK5ewLgNe773_SwmIqvTis_G diver3d_v2.zip
Processing file 1xYQCMGKUypaXu9aDUCKZK0rId2BjtTSJ diver3d_v2.zip.sha256


100%|██████████| 3.45G/3.45G [03:01<00:00, 19.0MB/s]
Downloading...
From: https://drive.google.com/uc?id=1xYQCMGKUypaXu9aDUCKZK0rId2BjtTSJ
To: /tmp/dl/diver3d_v2.zip.sha256
100%|██████████| 81.0/81.0 [00:00<00:00, 216kB/s]
Download completed


ROOT = /tmp/data/diver3d_v2


## Code (`d3` package)

In [4]:
import os
for d in ['d3','tests','tools']: os.makedirs(f'/kaggle/working/code/{d}', exist_ok=True)

In [6]:
%%writefile /kaggle/working/code/d3/__init__.py
# d3 package

Writing /kaggle/working/code/d3/__init__.py


In [7]:
%%writefile /kaggle/working/code/d3/geometry.py
"""
Diver3D geometry conventions -- the single source of truth.

Every other module imports from here. If a convention is wrong, it is wrong
in exactly one place, and tests/test_geometry.py will catch it.

CONVENTIONS (from FORMAT.md / the challenge brief)
- Sonar frame: right-handed, metres. x forward, y left, z up.
- Box = (center c, dims d=[l,w,h], rotation R).
- R = [ex ey ez] : the body axes are the COLUMNS of R, written in sonar coords.
    ex = chest normal         (extent l)
    ey = right -> left shoulder (extent w)
    ez = feet -> head         (extent h),  ez = ex x ey
  So a point p in the sonar frame has body coords  R^T (p - c),
  and a body-frame point b maps to sonar coords   c + R b.
- Quaternions are Hamilton (w, x, y, z).
- Metric symmetry: only S = Rz(pi) acting in the BODY frame is free,
  i.e. R and R @ diag(-1,-1,1) = [-ex, -ey, ez] are scored identically.
  Head<->feet (R @ Rx(pi) = [ex, -ey, -ez]) is a full 180 deg error.
"""
import numpy as np

SYM = np.diag([-1.0, -1.0, 1.0])                              # Rz(pi), body frame
MIRROR_Y = np.diag([1.0, -1.0, 1.0])                          # y -> -y reflection


# --------------------------------------------------------------------------- quaternions
def quat_to_R(q):
    """(..., 4) Hamilton (w,x,y,z) -> (..., 3, 3). Same formula as FORMAT.md."""
    q = np.asarray(q, dtype=np.float64)
    q = q / np.linalg.norm(q, axis=-1, keepdims=True)
    w, x, y, z = np.moveaxis(q, -1, 0)
    R = np.stack([
        1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y),
        2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x),
        2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y),
    ], axis=-1)
    return R.reshape(q.shape[:-1] + (3, 3))


def R_to_quat(R):
    """(..., 3, 3) -> (..., 4) Hamilton (w,x,y,z), canonicalised to w >= 0.
    Uses scipy (which returns x,y,z,w -- we reorder explicitly)."""
    from scipy.spatial.transform import Rotation
    R = np.asarray(R, dtype=np.float64)
    shp = R.shape[:-2]
    xyzw = Rotation.from_matrix(R.reshape(-1, 3, 3)).as_quat()
    q = xyzw[:, [3, 0, 1, 2]]
    q = q * np.where(q[:, :1] < 0, -1.0, 1.0)  # q and -q are the same rotation
    return q.reshape(shp + (4,))


# --------------------------------------------------------------------------- SO(3) helpers
def project_to_SO3(M):
    """Nearest rotation (Frobenius) to any 3x3 matrix, via SVD. Used for averaging."""
    U, _, Vt = np.linalg.svd(M)
    D = np.ones(M.shape[:-1])
    D[..., -1] = np.sign(np.linalg.det(U @ Vt))
    return (U * D[..., None, :]) @ Vt


def rot6d_zx_to_R(a, b):
    """Our INTERNAL 6D rotation: a ~ ez (head axis), b ~ ex (chest), Gram-Schmidt.
    Gram-Schmidt keeps the FIRST vector's direction exactly and only bends the
    second, so we put first the axis that matters most (ez: tilt + head-up).
    Returns R = [ex, ey, ez]."""
    a, b = np.asarray(a, float), np.asarray(b, float)
    ez = a / np.linalg.norm(a, axis=-1, keepdims=True)
    ex = b - np.sum(ez * b, axis=-1, keepdims=True) * ez
    ex = ex / np.linalg.norm(ex, axis=-1, keepdims=True)
    ey = np.cross(ez, ex)                # z x x = y  (right-handed, cyclic)
    return np.stack([ex, ey, ez], axis=-1)


def R_to_rot6d_zx(R):
    """Inverse of rot6d_zx_to_R: returns concat(ez, ex), shape (..., 6)."""
    R = np.asarray(R, float)
    return np.concatenate([R[..., :, 2], R[..., :, 0]], axis=-1)


def rot_angle_deg(R1, R2):
    """Plain geodesic angle between rotations, degrees."""
    tr = np.trace(np.swapaxes(R1, -1, -2) @ R2, axis1=-2, axis2=-1)
    return np.degrees(np.arccos(np.clip((tr - 1) / 2, -1, 1)))


def sym_angle_deg(R_pred, R_gt):
    """AOE3D as the metric defines it: min over S in {I, Rz(pi)} of angle(Rp^T Rg S)."""
    return np.minimum(rot_angle_deg(R_pred, R_gt), rot_angle_deg(R_pred, R_gt @ SYM))


def align_to_reference(R, R_ref):
    """Return R or R@SYM, whichever is closer to R_ref. Needed before averaging
    rotations (ensemble fusion, temporal smoothing): otherwise averaging R and its
    symmetric twin gives garbage."""
    flip = rot_angle_deg(R_ref, R @ SYM) < rot_angle_deg(R_ref, R)
    return np.where(np.asarray(flip)[..., None, None], R @ SYM, R)


def tilt_deg(R):
    """Angle between body up-axis ez and sonar z. 0 = upright, 90 = horizontal,
    180 = head down."""
    R = np.asarray(R, float)
    return np.degrees(np.arccos(np.clip(R[..., 2, 2], -1, 1)))


def Rz(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[c, -s, 0], [s, c, 0], [0, 0, 1.0]])


def Rx(theta):
    c, s = np.cos(theta), np.sin(theta)
    return np.array([[1.0, 0, 0], [0, c, -s], [0, s, c]])


# --------------------------------------------------------------------------- boxes
_SIGNS = np.array([[sx, sy, sz] for sx in (-1, 1) for sy in (-1, 1) for sz in (-1, 1)], float)


def box_corners(c, d, R):
    """8 corners in sonar frame: c + R @ (signs * d/2)."""
    return (_SIGNS * (np.asarray(d) / 2)) @ np.asarray(R).T + np.asarray(c)


def points_in_box(pts, c, d, R, margin=0.0):
    """Boolean mask of points (N,>=3) inside the oriented box (optionally enlarged)."""
    local = (np.asarray(pts)[:, :3] - np.asarray(c)) @ np.asarray(R)  # = R^T (p - c), row form
    return np.all(np.abs(local) <= np.asarray(d) / 2 + margin, axis=1)


Writing /kaggle/working/code/d3/geometry.py


In [8]:
%%writefile /kaggle/working/code/d3/transforms.py
"""
Joint point/box transforms used by augmentation and test-time augmentation (TTA).

Boxes are kept as arrays:  c (K,3) centers, d (K,3) dims [l,w,h], R (K,3,3).
Each function returns NEW arrays (never modifies inputs in place).

Rule for a proper rotation T (det +1) applied to the whole scene:
    p' = T p,  c' = T c,  R' = T R,  d' = d
Rule for the mirror M = diag(1,-1,1) (det -1, a reflection):
    p' = M p,  c' = M c,  R' = M R M,  d' = d
  Why M R M and not M R?  M R has det -1 (not a rotation). Mirroring the body
  swaps its left and right shoulders, so the new ey (right->left shoulder)
  is -(M ey). Negating the 2nd column is exactly right-multiplying by M:
    R' = [M ex, -M ey, M ez] = M R M,  and  ex' x ey' = M ez = ez'.
"""
import numpy as np
from .geometry import MIRROR_Y, Rz


def mirror_y(pts, c, R):
    pts = pts.copy(); pts[:, 1] = -pts[:, 1]
    c = c * np.array([1.0, -1.0, 1.0])
    R = MIRROR_Y @ R @ MIRROR_Y
    return pts, c, R


def rotate_z(pts, c, R, theta):
    """Rotate the scene about the SONAR z-axis through the sonar origin.
    This keeps each point's range and elevation and only changes bearing,
    so the sonar's view of the diver stays physically plausible."""
    T = Rz(theta)
    pts = pts.copy(); pts[:, :3] = pts[:, :3] @ T.T
    return pts, c @ T.T, T @ R


def scale(pts, c, d, s):
    pts = pts.copy(); pts[:, :3] *= s
    return pts, c * s, d * s


def mirror_boxes_only(c, R):
    """Undo a mirror on PREDICTIONS (for mirror TTA): the mirror is its own inverse."""
    return c * np.array([1.0, -1.0, 1.0]), MIRROR_Y @ R @ MIRROR_Y


Writing /kaggle/working/code/d3/transforms.py


In [9]:
%%writefile /kaggle/working/code/d3/rot_torch.py
"""
Torch rotation utilities for the network head and the loss.

Representation: the head outputs 6 numbers (a, b) ~ (ez, ex); Gram-Schmidt
turns them into R = [ex, ey, ez]. Identical to geometry.rot6d_zx_to_R.

Loss: symmetric CHORDAL loss, matching the metric's symmetry
    L = min( ||Rp - Rg||_F^2 , ||Rp - Rg S||_F^2 ),   S = diag(-1,-1,1)
Chordal ||A-B||^2 = 8 sin^2(theta/2) is monotonic in the geodesic angle
theta, so it has the same minimiser and the same choice of S. Unlike acos(),
its gradient is smooth everywhere (acos blows up near 0 deg and is zero after
clamping, which is why the old rot loss jumped around).
"""
import torch
import torch.nn.functional as F


def rot6d_zx_to_R(r6):
    """(K,6) -> (K,3,3) columns [ex, ey, ez]."""
    ez = F.normalize(r6[:, :3], dim=1, eps=1e-6)
    b = r6[:, 3:]
    ex = F.normalize(b - (ez * b).sum(1, keepdim=True) * ez, dim=1, eps=1e-6)
    ey = torch.cross(ez, ex, dim=1)
    return torch.stack([ex, ey, ez], dim=2)


def sym_chordal_loss(Rp, Rg):
    """(K,3,3),(K,3,3) -> (K,) per-box loss, in [0, 8]."""
    d_id = ((Rp - Rg) ** 2).sum((1, 2))
    sgn = Rg.new_tensor([-1.0, -1.0, 1.0])          # Rg @ S negates columns ex, ey
    d_sym = ((Rp - Rg * sgn) ** 2).sum((1, 2))
    return torch.minimum(d_id, d_sym)


def sym_angle_deg(Rp, Rg):
    """For logging only (not for backprop)."""
    def ang(A, B):
        tr = (A.transpose(1, 2) @ B).diagonal(dim1=1, dim2=2).sum(-1)
        return torch.rad2deg(torch.acos(((tr - 1) / 2).clamp(-1, 1)))
    sgn = Rg.new_tensor([-1.0, -1.0, 1.0])
    return torch.minimum(ang(Rp, Rg), ang(Rp, Rg * sgn))


Writing /kaggle/working/code/d3/rot_torch.py


In [10]:
%%writefile /kaggle/working/code/d3/io.py
"""
Data I/O: point clouds, GT, and submission writing.

GT / predictions are held in memory as  {token: dict(c=(K,3), d=(K,3), R=(K,3,3),
score=(K,), [iid=(K,)])}  -- arrays, not lists of dicts.
"""
import gzip
import json
import os
import numpy as np
from .geometry import quat_to_R, R_to_quat


def frame_path(root, token):
    scene, frame = token.split("/")
    return os.path.join(root, "scenes", scene, "sonar", f"frame_{int(frame):06d}.bin")


def load_bin(path):
    raw = np.fromfile(path, dtype=np.float32)
    assert raw.size % 4 == 0, path
    return raw.reshape(-1, 4)


def _empty():
    return dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)), R=np.zeros((0, 3, 3)),
                score=np.zeros(0), iid=np.zeros(0, int))


def load_gt(path):
    """Returns {token: arrays} in the file's order (= acquisition order)."""
    res = json.load(open(path))["results"]
    out = {}
    for tok, boxes in res.items():
        if not boxes:
            out[tok] = _empty(); continue
        out[tok] = dict(
            c=np.array([b["center"] for b in boxes], float),
            d=np.array([b["dimensions"] for b in boxes], float),
            R=quat_to_R(np.array([b["quaternion"] for b in boxes], float)),
            score=np.ones(len(boxes)),
            iid=np.array([b.get("instance_id", -1) for b in boxes], int))
    return out


def sort_tokens(tokens):
    """Acquisition order: by scene, then by frame number."""
    return sorted(tokens, key=lambda t: (t.split("/")[0], int(t.split("/")[1])))


def write_submission(preds, path, tokens=None, max_det=100, ndigits=5):
    """preds: {token: arrays}. Writes .json (dict) or .jsonl.gz (if path ends .gz).
    Every token in `tokens` is written (empty list if no prediction)."""
    tokens = tokens if tokens is not None else list(preds.keys())
    r = lambda a: [round(float(v), ndigits) for v in a]

    def boxes_of(tok):
        p = preds.get(tok)
        if p is None or len(p["c"]) == 0:
            return []
        order = np.argsort(-p["score"])[:max_det]
        q = R_to_quat(p["R"][order])
        d = np.maximum(p["d"][order], 1e-3)          # dims must be strictly positive
        s = np.clip(p["score"][order], 0.0, 1.0)
        return [dict(center=r(p["c"][i]), dimensions=r(d[k]), quaternion=r(q[k]),
                     score=round(float(s[k]), 6)) for k, i in enumerate(order)]

    if path.endswith(".gz"):
        with gzip.open(path, "wt") as f:
            for tok in tokens:
                f.write(json.dumps({"sample_token": tok, "boxes": boxes_of(tok)}) + "\n")
    else:
        json.dump({"results": {tok: boxes_of(tok) for tok in tokens}}, open(path, "w"))
    return path


Writing /kaggle/working/code/d3/io.py


In [11]:
%%writefile /kaggle/working/code/d3/evaluate.py
"""Thin wrapper around the OFFICIAL scorer so every number we quote is the server's."""
import os
import re
import subprocess
import sys


def run_scorer(root, pred_path, gt_path=None, verbose=False):
    gt_path = gt_path or os.path.join(root, "annotations", "gt_val.json")
    out = subprocess.run([sys.executable, os.path.join(root, "tools", "diver3d_eval.py"),
                          "--gt", gt_path, "--pred", pred_path],
                         capture_output=True, text=True)
    if out.returncode != 0:
        raise RuntimeError(out.stderr[-3000:])
    if verbose:
        print(out.stdout[-1200:])
    m = dict(re.findall(r"^\s+([A-Za-z0-9_@.]+)\s+(-?[\d.]+(?:e-?\d+)?)\s*$", out.stdout, re.M))
    return {k: float(v) for k, v in m.items()}


def fmt(m):
    keys = ["SDS", "mAP3D", "AP3D@0.50", "APBEV@0.35", "ATE_m", "ASE", "AOE3D_deg", "TILT_deg"]
    return "  ".join(f"{k} {m[k]:.4f}" for k in keys if k in m)


Writing /kaggle/working/code/d3/evaluate.py


In [12]:
%%writefile /kaggle/working/code/tests/test_geometry.py
"""
Convention unit tests. Run:  python -m pytest -q tests/test_geometry.py
Needs no data. Every test states the fact it checks.
"""
import os, sys, json, tempfile
import numpy as np
import pytest
import torch

sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.geometry import (quat_to_R, R_to_quat, rot6d_zx_to_R, R_to_rot6d_zx, SYM,
                         MIRROR_Y, sym_angle_deg, rot_angle_deg, tilt_deg, Rz, Rx,
                         box_corners, points_in_box, project_to_SO3, align_to_reference)
from d3 import transforms as T
from d3 import rot_torch
from d3.io import write_submission

rng = np.random.default_rng(0)


def rand_R(n):
    q = rng.normal(size=(n, 4))
    return quat_to_R(q)


def rand_box():
    c = rng.uniform([1, -4, -2], [10, 4, 2])
    d = rng.uniform([0.6, 0.6, 1.2], [1.2, 1.4, 2.1])
    return c, d, rand_R(1)[0]


def same_point_set(A, B, tol=1e-9):
    """Corners can come out in a different order; compare as sets."""
    D = np.linalg.norm(A[:, None] - B[None], axis=-1)
    return np.all(D.min(1) < tol) and np.all(D.min(0) < tol)


# ----------------------------------------------------------------- quaternions
def test_quat_order_is_wxyz():
    # 90 deg about z: q = (cos45, 0, 0, sin45). Must send x-axis to y-axis.
    R = quat_to_R([np.cos(np.pi / 4), 0, 0, np.sin(np.pi / 4)])
    assert np.allclose(R[:, 0], [0, 1, 0]) and np.allclose(R[:, 2], [0, 0, 1])


def test_quat_roundtrip_and_sign():
    R = rand_R(1000)
    q = R_to_quat(R)
    assert np.all(q[:, 0] >= 0)                      # canonical sign
    assert np.allclose(quat_to_R(q), R, atol=1e-9)
    assert np.allclose(quat_to_R(-q), R, atol=1e-9)  # q and -q are the same rotation


def test_rotations_are_proper():
    R = rand_R(1000)
    assert np.allclose(np.linalg.det(R), 1)
    assert np.allclose(np.swapaxes(R, 1, 2) @ R, np.eye(3), atol=1e-9)


def test_columns_are_body_axes():
    # ez = ex x ey  (right-handed body frame)
    R = rand_R(100)
    assert np.allclose(np.cross(R[:, :, 0], R[:, :, 1]), R[:, :, 2], atol=1e-9)


# ----------------------------------------------------------------- 6D
def test_rot6d_roundtrip_numpy_and_torch():
    R = rand_R(500)
    r6 = R_to_rot6d_zx(R)
    assert np.allclose(rot6d_zx_to_R(r6[:, :3], r6[:, 3:]), R, atol=1e-9)
    Rt = rot_torch.rot6d_zx_to_R(torch.tensor(r6)).numpy()
    assert np.allclose(Rt, R, atol=1e-9)


def test_rot6d_keeps_ez_exactly():
    # Gram-Schmidt must keep the FIRST vector (ez) direction, only bend ex.
    a, b = rng.normal(size=(100, 3)), rng.normal(size=(100, 3))
    R = rot6d_zx_to_R(a, b)
    assert np.allclose(R[:, :, 2], a / np.linalg.norm(a, axis=1, keepdims=True))
    assert np.allclose(np.linalg.det(R), 1)


# ----------------------------------------------------------------- metric symmetry
def test_symmetry_is_free_headflip_is_not():
    R = rand_R(200)
    assert np.allclose(sym_angle_deg(R @ SYM, R), 0, atol=1e-5)        # Rz(pi): free
    assert np.allclose(sym_angle_deg(R @ Rx(np.pi), R), 180, atol=1e-4)  # head<->feet
    # SYM is Rz(pi) in the body frame: ex,ey flip, ez unchanged
    assert np.allclose((R @ SYM)[:, :, 2], R[:, :, 2])
    assert np.allclose((R @ SYM)[:, :, :2], -R[:, :, :2])


def test_symmetric_box_is_same_cuboid():
    c, d, R = rand_box()
    assert same_point_set(box_corners(c, d, R), box_corners(c, d, R @ SYM))


def test_torch_loss_symmetric_and_matches_angle():
    R = torch.tensor(rand_R(300))
    Rs = R * R.new_tensor([-1.0, -1.0, 1.0])
    assert torch.allclose(rot_torch.sym_chordal_loss(R, Rs), torch.zeros(300, dtype=R.dtype), atol=1e-9)
    R2 = torch.tensor(rand_R(300))
    ang_np = sym_angle_deg(R2.numpy(), R.numpy())
    ang_t = rot_torch.sym_angle_deg(R2, R).numpy()
    assert np.allclose(ang_np, ang_t, atol=1e-4)
    # chordal = 8 sin^2(theta/2) at the symmetric minimum
    L = rot_torch.sym_chordal_loss(R2, R).numpy()
    assert np.allclose(L, 8 * np.sin(np.radians(ang_np) / 2) ** 2, atol=1e-6)


def test_torch_loss_has_gradient_at_zero_error():
    R = torch.tensor(rand_R(4))
    r6 = torch.tensor(R_to_rot6d_zx(R.numpy())) + 1e-3
    r6.requires_grad_(True)
    rot_torch.sym_chordal_loss(rot_torch.rot6d_zx_to_R(r6), R).sum().backward()
    assert torch.isfinite(r6.grad).all()


def test_align_to_reference():
    R = rand_R(50)
    R_ref = R @ SYM @ Rz(0.05)        # close to the symmetric twin
    A = align_to_reference(R, R_ref)
    assert np.all(rot_angle_deg(A, R_ref) < 5)


def test_project_to_SO3():
    R = rand_R(10)
    assert np.allclose(project_to_SO3(R + 1e-3 * rng.normal(size=R.shape)), R, atol=1e-2)
    # chordal mean of a rotation and itself is itself
    assert np.allclose(project_to_SO3((R + R) / 2), R, atol=1e-9)


# ----------------------------------------------------------------- augmentation transforms
def scene_with_box(n=4000):
    c, d, R = rand_box()
    pts = np.c_[rng.uniform([0, -5, -2.5], [12, 5, 2.5], size=(n, 3)), rng.uniform(size=n)]
    inside = (rng.uniform(-0.5, 0.5, size=(500, 3)) * d) @ R.T + c   # points inside the box
    pts = np.r_[pts, np.c_[inside, rng.uniform(size=500)]]
    return pts, c[None], d[None], R[None]


@pytest.mark.parametrize("which", ["mirror", "rotz", "scale"])
def test_transform_moves_box_with_points(which):
    pts, c, d, R = scene_with_box()
    m0 = points_in_box(pts, c[0], d[0], R[0])
    if which == "mirror":
        p2, c2, R2 = T.mirror_y(pts, c, R); d2 = d
        Tm = MIRROR_Y
    elif which == "rotz":
        th = 0.3; p2, c2, R2 = T.rotate_z(pts, c, R, th); d2 = d
        Tm = Rz(th)
    else:
        s = 1.04; p2, c2, d2 = T.scale(pts, c, d, s); R2 = R
        Tm = s * np.eye(3)
    # 1) exactly the same points are inside the box
    assert np.array_equal(m0, points_in_box(p2, c2[0], d2[0], R2[0]))
    # 2) the box is the same physical cuboid (transformed corners as a set)
    assert same_point_set(box_corners(c2[0], d2[0], R2[0]), box_corners(c[0], d[0], R[0]) @ Tm.T)
    # 3) still a proper rotation
    assert np.isclose(np.linalg.det(R2[0]), 1)
    # 4) inputs untouched
    assert not np.shares_memory(p2, pts)


def test_mirror_semantics():
    _, c, d, R = scene_with_box()
    _, _, R2 = T.mirror_y(np.zeros((1, 4)), c, R)
    R, R2 = R[0], R2[0]
    assert np.allclose(R2[:, 2], MIRROR_Y @ R[:, 2])   # head still points to mirrored head
    assert np.allclose(R2[:, 0], MIRROR_Y @ R[:, 0])   # chest normal mirrored
    assert np.allclose(R2[:, 1], -MIRROR_Y @ R[:, 1])  # left/right shoulders swap
    assert np.isclose(tilt_deg(R2), tilt_deg(R))       # mirror can't change tilt
    # the naive version M @ R is NOT a rotation -- this is the classic bug
    assert np.isclose(np.linalg.det(MIRROR_Y @ R), -1)


def test_mirror_is_involution():
    pts, c, d, R = scene_with_box()
    p2, c2, R2 = T.mirror_y(*T.mirror_y(pts, c, R))
    assert np.allclose(p2, pts) and np.allclose(c2, c) and np.allclose(R2, R)
    c3, R3 = T.mirror_boxes_only(*T.mirror_boxes_only(c, R))
    assert np.allclose(c3, c) and np.allclose(R3, R)


# ----------------------------------------------------------------- submission writer
@pytest.mark.parametrize("ext", [".json", ".jsonl.gz"])
def test_writer_roundtrip(ext):
    import gzip
    R = rand_R(3)
    preds = {"scene_0001/000000": dict(c=rng.normal(size=(3, 3)), d=np.ones((3, 3)),
                                       R=R, score=np.array([0.2, 0.9, 0.5])),
             "scene_0001/000001": dict(c=np.zeros((0, 3)), d=np.zeros((0, 3)),
                                       R=np.zeros((0, 3, 3)), score=np.zeros(0))}
    path = os.path.join(tempfile.mkdtemp(), "sub" + ext)
    write_submission(preds, path, tokens=list(preds) + ["scene_0001/000002"])
    if ext == ".json":
        res = json.load(open(path))["results"]
    else:
        res = {j["sample_token"]: j["boxes"] for j in map(json.loads, gzip.open(path, "rt"))}
    assert set(res) == {"scene_0001/000000", "scene_0001/000001", "scene_0001/000002"}
    b = res["scene_0001/000000"]
    assert [x["score"] for x in b] == [0.9, 0.5, 0.2]          # sorted by score
    order = [1, 2, 0]
    for k, i in enumerate(order):
        assert rot_angle_deg(quat_to_R(b[k]["quaternion"]), R[i]) < 1e-3
    assert res["scene_0001/000002"] == []


Writing /kaggle/working/code/tests/test_geometry.py


In [13]:
%%writefile /kaggle/working/code/tests/check_with_scorer.py
"""
Use the OFFICIAL scorer as the oracle for our conventions, and measure how
sensitive the metric is to each kind of error. Needs the dataset.

    python tests/check_with_scorer.py --root /kaggle/input/<...>/diver3d_v2

Part A (must pass): perfect GT written through OUR reader/writer scores 1.0, the
Rz(pi) twin is free, the head<->feet flip costs exactly 180 deg, and a simulated
quaternion-order bug is caught.

Part B (for understanding): perturb the perfect GT by one error type at a time
and watch what the metric does. This tells us which errors to fight.
"""
import argparse, os, sys, copy
import numpy as np
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.io import load_gt, write_submission
from d3.geometry import SYM, Rx, Rz, quat_to_R, R_to_quat
from d3 import transforms as T
from d3.evaluate import run_scorer, fmt

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
ap.add_argument("--out", default="/kaggle/working/scorer_checks")
ap.add_argument("--skip_sensitivity", action="store_true")
args = ap.parse_args()
os.makedirs(args.out, exist_ok=True)

gt = load_gt(os.path.join(args.root, "annotations", "gt_val.json"))
tokens = list(gt.keys())


def score(name, fn):
    p = {t: fn(copy.deepcopy(v)) for t, v in gt.items()}
    path = write_submission(p, os.path.join(args.out, name + ".json"), tokens)
    m = run_scorer(args.root, path)
    print(f"{name:28s} {fmt(m)}")
    return m


def apply_R(f):
    def g(b):
        if len(b["R"]): b["R"] = f(b["R"])
        return b
    return g


print("== Part A: convention checks against the official scorer ==")
m = score("identity", lambda b: b)
assert abs(m["SDS"] - 1) < 1e-3, "our reader/writer is broken"
m = score("rz_pi_twin", apply_R(lambda R: R @ SYM))
assert m["AOE3D_deg"] < 0.01 and abs(m["SDS"] - 1) < 1e-3, "Rz(pi) should be free"
m = score("head_feet_flip", apply_R(lambda R: R @ Rx(np.pi)))
assert abs(m["AOE3D_deg"] - 180) < 0.1, "head flip should cost 180 deg"
m = score("quat_order_bug_xyzw", apply_R(lambda R: quat_to_R(R_to_quat(R)[:, [1, 2, 3, 0]])))
assert m["SDS"] < 0.9, "scorer should punish wrong quaternion order"


def mirror_twice(b):
    if len(b["c"]):
        _, c, R = T.mirror_y(np.zeros((1, 4)), b["c"], b["R"])
        _, b["c"], b["R"] = T.mirror_y(np.zeros((1, 4)), c, R)
    return b
m = score("mirror_twice", mirror_twice)
assert abs(m["SDS"] - 1) < 1e-3
print("Part A passed: conventions agree with the official scorer.\n")

if args.skip_sensitivity:
    sys.exit()

print("== Part B: metric sensitivity (one error type at a time) ==")
def shift(axis, delta):
    def f(b):
        if len(b["c"]): b["c"][:, axis] += delta
        return b
    return f
def scale_dim(k, s):
    def f(b):
        if len(b["d"]): b["d"][:, k] *= s
        return b
    return f
for dz in (0.1, 0.2, 0.3):
    score(f"z +{dz} m", shift(2, dz))
for dx in (0.1, 0.2, 0.3):
    score(f"x +{dx} m (range)", shift(0, dx))
for s in (0.85, 1.15):
    score(f"height x{s}", scale_dim(2, s))
    score(f"length x{s}", scale_dim(0, s))
for deg in (10, 20, 30):
    score(f"tilt about ex {deg} deg", apply_R(lambda R, a=np.radians(deg): R @ Rx(a)))
    score(f"yaw about ez {deg} deg", apply_R(lambda R, a=np.radians(deg): R @ Rz(a)))


Writing /kaggle/working/code/tests/check_with_scorer.py


In [14]:
%%writefile /kaggle/working/code/tools/data_report.py
"""
Answers the planning questions with numbers instead of guesses (~3-5 min, CPU).

    python tools/data_report.py --root /kaggle/input/<...>/diver3d_v2

Q1 Pose mix per split, incl. head-down share       -> rebalancing, head-flip head?
Q2 Is the sonar static between frames?              -> is multi-frame input worth it?
Q3 Are frames contiguous, and how fast do divers move? -> tracking/Kalman settings
Q4 How stable are box dims within one track?        -> size smoothing / priors
Q5 How much GT does each crop lose?                 -> crop choice
"""
import argparse, os, sys, collections
import numpy as np
from scipy.spatial import cKDTree
sys.path.insert(0, os.path.dirname(os.path.dirname(os.path.abspath(__file__))))
from d3.io import load_gt, load_bin, frame_path, sort_tokens
from d3.geometry import tilt_deg, box_corners

ap = argparse.ArgumentParser()
ap.add_argument("--root", required=True)
args = ap.parse_args()
rng = np.random.default_rng(0)
G = {s: load_gt(os.path.join(args.root, "annotations", f"gt_{s}.json")) for s in ("train", "val")}

print("\n== Q1 pose mix (tilt = angle between body head-axis and sonar up) ==")
for s, gt in G.items():
    t = np.concatenate([tilt_deg(v["R"]) for v in gt.values() if len(v["c"])])
    print(f"{s:5s} n={len(t):5d} upright<15 {np.mean(t<15)*100:5.1f}%  15-75 {np.mean((t>=15)&(t<75))*100:5.1f}%"
          f"  horiz75-105 {np.mean((t>=75)&(t<=105))*100:5.1f}%  head-down>105 {np.mean(t>105)*100:5.1f}%")
    n = collections.Counter(len(v["c"]) for v in gt.values())
    print(f"      divers/frame: {dict(sorted(n.items()))}")

print("\n== Q2 is the sonar static? (fraction of frame-t points within 5 cm of frame t-1) ==")
tr = sort_tokens(G["train"].keys())
def overlap(a, b):
    A = load_bin(frame_path(args.root, a))[:, :3]; B = load_bin(frame_path(args.root, b))[:, :3]
    d, _ = cKDTree(B).query(A, k=1)
    return np.mean(d < 0.05), np.median(d)
same, diff = [], []
for _ in range(40):
    i = rng.integers(1, len(tr))
    if tr[i].split("/")[0] == tr[i - 1].split("/")[0]:
        same.append(overlap(tr[i], tr[i - 1]))
    j = rng.integers(len(tr))                       # control: a far-away frame
    if abs(j - i) > 100:
        diff.append(overlap(tr[i], tr[j]))
same, diff = np.array(same), np.array(diff)
print(f"consecutive frames : {same[:,0].mean()*100:5.1f}% within 5cm, median NN dist {np.median(same[:,1])*100:.1f} cm")
if len(diff): print(f"far-apart frames   : {diff[:,0].mean()*100:5.1f}% within 5cm, median NN dist {np.median(diff[:,1])*100:.1f} cm  (control)")
print("-> if consecutive >> control, background is stable and stacking frames can help.")

print("\n== Q3 frame contiguity and diver motion ==")
for s, gt in G.items():
    toks = sort_tokens(gt.keys()); gaps = collections.Counter()
    steps, rot_steps = [], []
    prev = {}
    for t in toks:
        sc, fr = t.split("/"); fr = int(fr)
        if sc in prev and fr - prev[sc][0] != 1: gaps[fr - prev[sc][0]] += 1
        v = gt[t]; cur = {int(i): (v["c"][k], v["R"][k]) for k, i in enumerate(v["iid"])}
        if sc in prev and fr - prev[sc][0] == 1:
            for i, (c, R) in cur.items():
                if i in prev[sc][1]:
                    steps.append(np.linalg.norm(c - prev[sc][1][i][0]))
                    tr_ = np.trace(prev[sc][1][i][1].T @ R)
                    rot_steps.append(np.degrees(np.arccos(np.clip((tr_ - 1) / 2, -1, 1))))
        prev[sc] = (fr, cur)
    print(f"{s:5s} frame gaps != 1: {dict(gaps) or 'none'}")
    print(f"      per-frame center step (m)  p50 {np.median(steps):.3f}  p90 {np.percentile(steps,90):.3f}  p99 {np.percentile(steps,99):.3f}")
    print(f"      per-frame rotation step (deg) p50 {np.median(rot_steps):.1f}  p90 {np.percentile(rot_steps,90):.1f}  p99 {np.percentile(rot_steps,99):.1f}")

print("\n== Q4 dims stability (std within a track vs across all boxes) ==")
for s, gt in G.items():
    tracks = collections.defaultdict(list)
    for t, v in gt.items():
        for k, i in enumerate(v["iid"]): tracks[(t.split('/')[0], int(i))].append(v["d"][k])
    allD = np.concatenate([np.array(x) for x in tracks.values()])
    within = np.mean([np.std(np.array(x), 0) for x in tracks.values() if len(x) > 20], 0)
    print(f"{s:5s} median dims {np.median(allD,0).round(2)}  global std {allD.std(0).round(3)}  within-track std {within.round(3)}")

print("\n== Q5 GT kept by each crop (center inside / all 8 corners inside) ==")
crops = {"paper [0,12]x[-5,5]x[-2.5,2.5]": ((0, 12), (-5, 5), (-2.5, 2.5)),
         "wider [0,14]x[-6,6]x[-3,3]":     ((0, 14), (-6, 6), (-3, 3))}
for s, gt in G.items():
    C = np.concatenate([v["c"] for v in gt.values() if len(v["c"])])
    K = np.concatenate([np.stack([box_corners(c, d, R) for c, d, R in zip(v["c"], v["d"], v["R"])])
                        for v in gt.values() if len(v["c"])])
    for name, (xr, yr, zr) in crops.items():
        inside = lambda P: ((P[..., 0] >= xr[0]) & (P[..., 0] < xr[1]) & (P[..., 1] >= yr[0]) &
                            (P[..., 1] < yr[1]) & (P[..., 2] >= zr[0]) & (P[..., 2] < zr[1]))
        print(f"{s:5s} {name:34s} centers {inside(C).mean()*100:6.2f}%   whole box {inside(K).all(1).mean()*100:6.2f}%")
    print(f"      center z range [{C[:,2].min():.2f}, {C[:,2].max():.2f}]   corner z range [{K[...,2].min():.2f}, {K[...,2].max():.2f}]")


Writing /kaggle/working/code/tools/data_report.py


## Unit tests

In [15]:
!cd /kaggle/working/code && pip -q install pytest && python -m pytest -q tests/test_geometry.py

...................                                                      [100%]
19 passed in 6.38s


## Convention checks with the official scorer + metric sensitivity

In [16]:
!cd /kaggle/working/code && python tests/check_with_scorer.py --root {ROOT}

== Part A: convention checks against the official scorer ==
identity                     SDS 1.0000  mAP3D 1.0000  AP3D@0.50 1.0000  APBEV@0.35 1.0000  ATE_m 0.0000  ASE 0.0000  AOE3D_deg 0.0005  TILT_deg 0.0003
rz_pi_twin                   SDS 1.0000  mAP3D 1.0000  AP3D@0.50 1.0000  APBEV@0.35 1.0000  ATE_m 0.0000  ASE 0.0000  AOE3D_deg 0.0005  TILT_deg 0.0003
head_feet_flip               SDS 0.8333  mAP3D 1.0000  AP3D@0.50 1.0000  APBEV@0.35 1.0000  ATE_m 0.0000  ASE 0.0000  AOE3D_deg 179.9997  TILT_deg 179.9997
quat_order_bug_xyzw          SDS 0.6920  mAP3D 0.7131  AP3D@0.50 0.3541  APBEV@0.35 0.9804  ATE_m 0.0000  ASE 0.0000  AOE3D_deg 88.8240  TILT_deg 82.2296
mirror_twice                 SDS 1.0000  mAP3D 1.0000  AP3D@0.50 1.0000  APBEV@0.35 1.0000  ATE_m 0.0000  ASE 0.0000  AOE3D_deg 0.0005  TILT_deg 0.0003
Part A passed: conventions agree with the official scorer.

== Part B: metric sensitivity (one error type at a time) ==
z +0.1 m                     SDS 0.9833  mAP3D 1.0000 

## Data report

In [17]:
!cd /kaggle/working/code && python tools/data_report.py --root {ROOT}


== Q1 pose mix (tilt = angle between body head-axis and sonar up) ==
train n=19482 upright<15  19.6%  15-75  28.2%  horiz75-105  40.8%  head-down>105  11.5%
      divers/frame: {0: 5556, 1: 11688, 2: 3223, 3: 228, 4: 166}
val   n= 7509 upright<15  52.1%  15-75  31.5%  horiz75-105  15.2%  head-down>105   1.2%
      divers/frame: {0: 2083, 1: 4782, 2: 829, 3: 303, 4: 40}

== Q2 is the sonar static? (fraction of frame-t points within 5 cm of frame t-1) ==
consecutive frames :  43.2% within 5cm, median NN dist 6.4 cm
far-apart frames   :   1.2% within 5cm, median NN dist 69.8 cm  (control)
-> if consecutive >> control, background is stable and stacking frames can help.

== Q3 frame contiguity and diver motion ==
train frame gaps != 1: none
      per-frame center step (m)  p50 0.069  p90 0.253  p99 0.617
      per-frame rotation step (deg) p50 0.0  p90 10.0  p99 35.7
val   frame gaps != 1: none
      per-frame center step (m)  p50 0.030  p90 0.205  p99 0.511
      per-frame rotation step (